[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch15-responsible-engr/02-cai-gia-cua-cong-bang.ipynb)

# Ba định nghĩa công bằng, và cái giá của việc chọn một

Theo chương 15, công bằng có nhiều định nghĩa toán học, và chúng có thể xung đột khi tỉ lệ nền khác
nhau và dự đoán không hoàn hảo. Chương dựng hai ví dụ: một mô hình duyệt vay với hai ma trận nhầm lẫn,
và một phép tính nhẩm về cái giá của công bằng trong tuyển dụng. Sổ tay này tính lại cả hai, rồi đi xa
hơn trên một mô hình thật: huấn luyện một bộ phân loại, đo từng định nghĩa trên từng nhóm, rồi dời
ngưỡng của từng nhóm để thấy đánh đổi hiện ra thành một đường biên Pareto đo được.

**Bạn sẽ làm:**
1. tính ba định nghĩa công bằng từ hai ma trận nhầm lẫn của bảng 6 và 7, và viết một bộ theo dõi báo
   "chưa đủ dữ liệu" khi một tỉ lệ không xác định;
2. xem vì sao bỏ thuộc tính nhạy cảm khỏi dữ liệu không đủ: một biến thay thế mang nó trở lại;
3. huấn luyện một mô hình tuyển dụng trên dữ liệu tổng hợp, đo mỗi nhóm ở một ngưỡng chung;
4. kiểm một đẳng thức nối các tỉ lệ lỗi với tỉ lệ nền, và thấy vì sao các tiêu chí không thể cùng đúng;
5. dời ngưỡng của từng nhóm và vẽ biên Pareto giữa độ chính xác và chênh lệch;
6. lặp lại phép tính nhẩm 1.2, rồi đo cùng đại lượng lợi ích ấy theo ngưỡng.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # nhóm A, nhóm B, và hai màu phụ
XAM = "0.45"

## 1. Hai ma trận nhầm lẫn, ba định nghĩa

Bảng 6 và bảng 7 là kết quả duyệt vay của cùng một mô hình, đánh giá riêng trên hai nhóm, trong một
kịch bản được dựng để minh hoạ. Mỗi ma trận có bốn ô: dương tính thật (TP, trả được nợ và được duyệt),
âm tính giả (FN, trả được nợ mà bị từ chối), dương tính giả (FP, vỡ nợ mà được duyệt) và âm tính thật
(TN).

In [ ]:
NHOM_A = dict(  # bảng 6, nhóm đa số
    tp=sach(4_500, nguon="10,000 · 5,000 · 4,500 · 500 · 1,000 · 4,000"),
    fn=sach(500, nguon="10,000 · 5,000 · 4,500 · 500 · 1,000 · 4,000"),
    fp=sach(1_000, nguon="10,000 · 5,000 · 4,500 · 500 · 1,000 · 4,000"),
    tn=sach(4_000, nguon="10,000 · 5,000 · 4,500 · 500 · 1,000 · 4,000"),
)
NHOM_B = dict(  # bảng 7, nhóm thiểu số
    tp=sach(600, nguon="2,000 · 1,000 · 600 · 400 · 200 · 800"),
    fn=sach(400, nguon="2,000 · 1,000 · 600 · 400 · 200 · 800"),
    fp=sach(200, nguon="2,000 · 1,000 · 600 · 400 · 200 · 800"),
    tn=sach(800, nguon="2,000 · 1,000 · 600 · 400 · 200 · 800"),
)


def chia(tu, mau):
    """Một tỉ lệ, hoặc None khi mẫu số bằng 0: tỉ lệ ấy không xác định, không phải bằng 0."""
    return tu / mau if mau else None


def chi_so_nhom(tp, fn, fp, tn):
    n = tp + fn + fp + tn
    return {
        "số người": n,
        "tỉ lệ nền": chia(tp + fn, n),
        "tỉ lệ được duyệt": chia(tp + fp, n),
        "TPR": chia(tp, tp + fn),
        "FPR": chia(fp, fp + tn),
        "FNR": chia(fn, tp + fn),
        "PPV": chia(tp, tp + fp),
        "độ chính xác": chia(tp + tn, n),
    }


cs_a, cs_b = chi_so_nhom(**NHOM_A), chi_so_nhom(**NHOM_B)
theo_sach("cỡ nhóm A", cs_a["số người"], sach=10_000, nguon="10,000 · 5,000 · 4,500")
theo_sach("cỡ nhóm B", cs_b["số người"], sach=2_000, nguon="2,000 · 1,000 · 600")
for ten in cs_a:
    if ten != "số người":
        print(f"{ten:>16}: nhóm A {cs_a[ten]:.4f} · nhóm B {cs_b[ten]:.4f}")

### Dự đoán

Ngang bằng nhân khẩu học (demographic parity) so tỉ lệ được duyệt. Cơ hội bình đẳng (equal opportunity)
so tỉ lệ dương tính thật (true positive rate, TPR), tức phần được duyệt trong số người sẽ trả được nợ.
Cân bằng xác suất (equalized odds) so cả TPR lẫn tỉ lệ dương tính giả (false positive rate, FPR). Với
bảng trên, định nghĩa nào thấy chênh lệch lớn nhất, và có định nghĩa nào thấy không chênh gì không?

In [ ]:
theo_sach("tỉ lệ được duyệt, nhóm A (%)", 100 * cs_a["tỉ lệ được duyệt"], sach=55, nguon="55 · 40 · 15 · 90 · 60")
theo_sach("tỉ lệ được duyệt, nhóm B (%)", 100 * cs_b["tỉ lệ được duyệt"], sach=40, nguon="55 · 40 · 15 · 90 · 60")
theo_sach("chênh lệch ngang bằng nhân khẩu học (điểm)",
          100 * (cs_a["tỉ lệ được duyệt"] - cs_b["tỉ lệ được duyệt"]), sach=15, nguon="55 · 40 · 15 · 90 · 60")
theo_sach("TPR, nhóm A (%)", 100 * cs_a["TPR"], sach=90, nguon="55 · 40 · 15 · 90 · 60 · 30")
theo_sach("TPR, nhóm B (%)", 100 * cs_b["TPR"], sach=60, nguon="55 · 40 · 15 · 90 · 60 · 30")
theo_sach("chênh lệch cơ hội bình đẳng (điểm)", 100 * (cs_a["TPR"] - cs_b["TPR"]), sach=30,
          nguon="55 · 40 · 15 · 90 · 60 · 30")
theo_sach("FPR, nhóm A (%)", 100 * cs_a["FPR"], sach=20, nguon="90 · 60 · 30 · 20 · 20 · 0")
theo_sach("FPR, nhóm B (%)", 100 * cs_b["FPR"], sach=20, nguon="90 · 60 · 30 · 20 · 20 · 0")
theo_sach("chênh lệch FPR (điểm)", 100 * (cs_a["FPR"] - cs_b["FPR"]), sach=0, nguon="90 · 60 · 30 · 20 · 20 · 0")
theo_sach("tỉ lệ âm tính giả, nhóm A (%)", 100 * cs_a["FNR"], sach=10, nguon="10 · 40 phần trăm")
theo_sach("tỉ lệ âm tính giả, nhóm B (%)", 100 * cs_b["FNR"], sach=40, nguon="10 · 40 phần trăm")
theo_sach("người đủ điều kiện nhóm B bị từ chối, so với nhóm A", cs_b["FNR"] / cs_a["FNR"], sach=4,
          nguon="30 điểm phần trăm · 4 lần")
theo_sach("độ chính xác, nhóm đa số (%)", 100 * cs_a["độ chính xác"], sach=85, nguon="85 · 82.5 phần trăm")
dung_tong = NHOM_A["tp"] + NHOM_A["tn"] + NHOM_B["tp"] + NHOM_B["tn"]
theo_sach("độ chính xác tổng thể (%)", 100 * dung_tong / (cs_a["số người"] + cs_b["số người"]), sach=82.5,
          nguon="85 · 82.5 phần trăm")
print(f"độ chính xác nhóm B (sách không in, suy ra từ bảng 7): {100 * cs_b['độ chính xác']:.0f} %")

Ba định nghĩa, ba kết luận, từ cùng tám con số: ngang bằng nhân khẩu học thấy 15 điểm, cơ hội bình
đẳng thấy 30 điểm, và riêng về FPR thì hai nhóm bằng hệt nhau. Cân bằng xác suất bị vi phạm vì TPR lệch.

Bảng còn cho thấy một điều nhỏ mà quan trọng cho phần 4: **hai nhóm có cùng tỉ lệ nền**, 50 phần trăm
(5,000 trên 10,000 và 1,000 trên 2,000). Kết quả bất khả của Kleinberg và cộng sự nói các tiêu chí xung
đột khi tỉ lệ nền khác nhau, nên ở ví dụ này nó chưa ràng buộc gì. Chênh lệch ở đây không đến từ tỉ lệ
nền, và hai ma trận cũng không cho biết nó đến từ đâu.

### Một bộ theo dõi công bằng

Listing 1 của chương tính chỉ số theo nhóm từ ma trận nhầm lẫn rồi báo động khi chênh lệch vượt ngưỡng,
với một hợp đồng rõ ràng: tỉ lệ nào không xác định thì báo "chưa đủ dữ liệu", không đem so với ngưỡng.
Bản dưới viết theo hợp đồng ấy, với ngưỡng phát hành 5 điểm phần trăm của kịch bản.

In [ ]:
NGUONG_PHAT_HANH = sach(5, nguon="5 điểm phần trăm") / 100


def theo_doi_cong_bang(nhom_1, nhom_2, nguong=NGUONG_PHAT_HANH):
    """So ba tỉ lệ giữa hai nhóm; mỗi dòng là 'ổn', 'BÁO ĐỘNG' hoặc 'chưa đủ dữ liệu'."""
    c1, c2 = chi_so_nhom(**nhom_1), chi_so_nhom(**nhom_2)
    bao_cao = {}
    for ten in ("tỉ lệ được duyệt", "TPR", "FPR"):
        if c1[ten] is None or c2[ten] is None:
            bao_cao[ten] = ("chưa đủ dữ liệu", None)
            continue
        lech = abs(c1[ten] - c2[ten])
        bao_cao[ten] = ("BÁO ĐỘNG" if lech > nguong else "ổn", lech)
    return bao_cao


for ten, (trang_thai, lech) in theo_doi_cong_bang(NHOM_A, NHOM_B).items():
    print(f"{ten:>16}: {trang_thai}" + (f", chênh {100 * lech:.0f} điểm" if lech is not None else ""))

# Một lát cắt mới ra mắt: 12 người nộp đơn, chưa ai trong số đó có kết quả "vỡ nợ".
lat_moi = dict(tp=7, fn=5, fp=0, tn=0)
print("lát cắt mới:", {k: v[0] for k, v in theo_doi_cong_bang(NHOM_A, lat_moi).items()})
assert theo_doi_cong_bang(NHOM_A, lat_moi)["FPR"][0] == "chưa đủ dữ liệu"

Khoảng cách 30 điểm vượt xa ngưỡng 5 điểm, nên theo sách, mô hình cần được can thiệp về công bằng trước
khi triển khai. Ở lát cắt mới, FPR có mẫu số bằng 0. Một đoạn mã viết vội sẽ coi nó là 0, rồi báo FPR
"ổn"; hợp đồng của listing 1 không cho làm thế. Sổ tay 01 đi tiếp ý này: ngay cả khi mẫu số khác 0
nhưng nhỏ, con số vẫn chưa đủ làm bằng chứng.

## 2. Bỏ thuộc tính nhạy cảm không đủ

Công cụ tuyển dụng của Amazon học từ một thập kỷ dữ liệu tuyển dụng lịch sử. Amazon bỏ các chỉ báo giới
tính tường minh, nhưng theo sách, những đặc trưng khác trong hồ sơ vẫn có thể giữ tín hiệu tương quan
với giới, như các trường hay câu lạc bộ. Đó là biến thay thế (proxy variable).

Sổ tay dựng một bài toán tuyển dụng tổng hợp để xem cơ chế ấy (mọi con số của dữ liệu là giả định):

* mỗi ứng viên có một năng lực ẩn $q$; kết quả thật $y$ (làm tốt công việc) phụ thuộc vào $q$ và chỉ vào
  $q$;
* hai đặc trưng `diem_1`, `diem_2` đo $q$ có nhiễu, như điểm bài kiểm tra và điểm phỏng vấn;
* đặc trưng `ma_hoat_dong` không nói gì về $q$, nhưng tương quan với nhóm, như tên một câu lạc bộ;
* trong phần này hai nhóm có **cùng** phân phối năng lực, nhưng nhãn huấn luyện là quyết định tuyển dụng
  lịch sử, và quá khứ đòi nhóm B một năng lực cao hơn mới tuyển.

In [ ]:
CO_A = sach(10_000, nguon="10,000 · 5,000 · 4,500")  # cùng cỡ hai nhóm như bảng 6 và 7
CO_B = sach(2_000, nguon="2,000 · 1,000 · 600")


def tao_ung_vien(r, lech_nang_luc=0.0, nhieu_a=0.6, nhieu_b=0.6, thien_lech_lich_su=0.0, he_so=1):
    """Dữ liệu tổng hợp: X (diem_1, diem_2, ma_hoat_dong), kết quả thật y, nhãn lịch sử, nhóm g."""
    n_a, n_b = CO_A * he_so, CO_B * he_so
    g = np.r_[np.zeros(n_a, int), np.ones(n_b, int)]
    q = r.normal(0, 1, n_a + n_b) - lech_nang_luc * g
    y = (r.random(len(q)) < 1 / (1 + np.exp(-2 * q))).astype(int)
    nhieu = np.where(g == 1, nhieu_b, nhieu_a)
    diem_1 = q + r.normal(0, 1, len(q)) * nhieu
    diem_2 = q + r.normal(0, 1, len(q)) * nhieu
    ma_hoat_dong = g + r.normal(0, 0.6, len(q))
    # Quyết định lịch sử: cùng năng lực, nhóm B bị trừ đi một khoảng.
    nhan_lich_su = (r.random(len(q)) < 1 / (1 + np.exp(-2 * (q - thien_lech_lich_su * g)))).astype(int)
    return np.c_[diem_1, diem_2, ma_hoat_dong], y, nhan_lich_su, g


X, y, y_ls, g = tao_ung_vien(np.random.default_rng(SEED), thien_lech_lich_su=0.8)
hl = np.random.default_rng(SEED).permutation(len(y))
hl, kt = hl[: len(y) // 2], hl[len(y) // 2:]
print(f"tỉ lệ làm tốt thật: nhóm A {y[g == 0].mean():.3f}, nhóm B {y[g == 1].mean():.3f}")
print(f"tỉ lệ được tuyển trong lịch sử: nhóm A {y_ls[g == 0].mean():.3f}, nhóm B {y_ls[g == 1].mean():.3f}")
do_tren_may("AUC đoán nhóm chỉ từ ma_hoat_dong", roc_auc_score(g, X[:, 2]))

### Dự đoán

Ba mô hình học từ nhãn lịch sử: một mô hình thấy cả cột nhóm, một mô hình đã bỏ cột nhóm nhưng còn
`ma_hoat_dong`, và một mô hình bỏ cả hai. Đo trên kết quả thật, tức trong số người sẽ làm tốt, mỗi mô
hình tuyển bao nhiêu phần trăm của từng nhóm? Mô hình thứ hai gần mô hình thứ nhất hay mô hình thứ ba?

In [ ]:
cac_cot = {
    "thấy cột nhóm": (np.c_[X, g], ["diem_1", "diem_2", "ma_hoat_dong", "nhom"]),
    "bỏ cột nhóm": (X, ["diem_1", "diem_2", "ma_hoat_dong"]),
    "bỏ cả biến thay thế": (X[:, :2], ["diem_1", "diem_2"]),
}
khoang_proxy = {}
for ten, (Xc, ten_cot) in cac_cot.items():
    m = LogisticRegression().fit(Xc[hl], y_ls[hl])
    chon = m.predict(Xc[kt]) == 1
    tpr = [chon[(g[kt] == G) & (y[kt] == 1)].mean() for G in (0, 1)]
    khoang_proxy[ten] = tpr[0] - tpr[1]
    he_so_cot = ", ".join(f"{c} {w:+.2f}" for c, w in zip(ten_cot, m.coef_[0]))
    print(f"{ten:>20}: hệ số [{he_so_cot}]")
    do_tren_may(f"   tỉ lệ tuyển người làm tốt, A − B ({ten})", 100 * (tpr[0] - tpr[1]), "điểm")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
ten_mo_hinh = list(khoang_proxy)
gia_tri = [100 * khoang_proxy[k] for k in ten_mo_hinh]
ax.barh(ten_mo_hinh, gia_tri, color=[MAU[1], MAU[3], MAU[2]], height=0.55)
for i, v in enumerate(gia_tri):
    ax.text(v + 0.6, i, f"{v:.1f} điểm", va="center", fontsize=8)
ax.axvline(100 * NGUONG_PHAT_HANH, color=XAM, lw=1, ls="--")
ax.text(100 * NGUONG_PHAT_HANH + 0.5, 0.5, "ngưỡng 5 điểm", va="center", fontsize=8, color=XAM)
ax.set_xlim(0, max(gia_tri) * 1.3)
ax.invert_yaxis()
ax.set_xlabel("chênh lệch tỉ lệ tuyển người làm tốt, A − B (điểm phần trăm)")
ax.set_title("Học từ nhãn lịch sử thiên lệch (đo trên dữ liệu tổng hợp)")
plt.tight_layout()
plt.show()

Mô hình thấy cột nhóm học thẳng thiên lệch của quá khứ: hệ số của cột nhóm âm. Bỏ cột ấy đi, mô hình
dồn một phần tín hiệu sang `ma_hoat_dong`, một đặc trưng tự nó không nói gì về năng lực: hệ số của nó
đi từ gần 0 sang âm, và chênh lệch nhỏ đi nhưng vẫn còn. Một biến thay thế chỉ mang lại một phần của
thuộc tính, theo mức nó tương quan với nhóm. Chỉ khi bỏ cả biến thay thế thì chênh lệch mới giảm hẳn,
và đó là trường hợp dễ: ở đây
sổ tay biết cột nào là biến thay thế. Trong hồ sơ thật, tín hiệu có thể rải trên nhiều đặc trưng, mỗi
đặc trưng một chút, nên theo sách, bỏ thuộc tính được bảo vệ tự nó không xác lập được công bằng.

Phần còn lại của sổ tay bỏ thiên lệch của nhãn đi, để nhìn vào một nguồn chênh lệch khác.

## 3. Một ngưỡng chung, hai nhóm

Từ đây, nhãn là kết quả thật. Hai giả định mới của sổ tay, chọn để các định nghĩa công bằng va nhau:

* **tỉ lệ nền khác nhau**: năng lực của nhóm B lệch xuống 0.6, nên ít người làm tốt hơn trong dữ liệu;
* **đặc trưng kém thông tin hơn ở nhóm B**: nhiễu đo 1.5 thay vì 0.6, như hồ sơ mỏng hơn.

Mô hình học trên một bộ dữ liệu cỡ của bảng 6 và 7. Ngưỡng được chọn trên một bộ kiểm định, và kết quả
báo trên một bộ kiểm tra; hai bộ này lớn gấp năm lần (`HE_SO_DANH_GIA`), để con số của nhóm B đủ chặt,
như sổ tay 01 đã tính.

In [ ]:
HE_SO_DANH_GIA = 5
CAU_HINH = dict(lech_nang_luc=0.6, nhieu_a=0.6, nhieu_b=1.5)
X_hl, y_hl, _, g_hl = tao_ung_vien(np.random.default_rng(SEED + 1), **CAU_HINH)
X_kd, y_kd, _, g_kd = tao_ung_vien(np.random.default_rng(SEED + 2), he_so=HE_SO_DANH_GIA, **CAU_HINH)
X_kt, y_kt, _, g_kt = tao_ung_vien(np.random.default_rng(SEED + 3), he_so=HE_SO_DANH_GIA, **CAU_HINH)
mo_hinh = LogisticRegression().fit(X_hl, y_hl)
diem_kd, diem_kt = mo_hinh.predict_proba(X_kd)[:, 1], mo_hinh.predict_proba(X_kt)[:, 1]
for G, ten in ((0, "A"), (1, "B")):
    print(f"nhóm {ten}: tỉ lệ nền {y_kt[g_kt == G].mean():.3f}")
    do_tren_may(f"   AUC trong nhóm {ten}", roc_auc_score(y_kt[g_kt == G], diem_kt[g_kt == G]))

NGUONG = np.round(np.arange(0.02, 0.99, 0.01), 2)  # lưới ngưỡng; dương khi điểm ≥ ngưỡng


def ti_le_theo_nguong(diem, nhan):
    """Mọi tỉ lệ của một nhóm, tại mọi ngưỡng của lưới."""
    tp = np.array([(diem[nhan == 1] >= t).sum() for t in NGUONG])
    fp = np.array([(diem[nhan == 0] >= t).sum() for t in NGUONG])
    P, N = int((nhan == 1).sum()), int((nhan == 0).sum())
    return dict(tpr=tp / P, fpr=fp / N, chon=(tp + fp) / (P + N), dung=tp + N - fp, n=P + N, nen=P / (P + N),
                ppv=np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), np.nan))


R_kd = {G: ti_le_theo_nguong(diem_kd[g_kd == G], y_kd[g_kd == G]) for G in (0, 1)}
R_kt = {G: ti_le_theo_nguong(diem_kt[g_kt == G], y_kt[g_kt == G]) for G in (0, 1)}
i50 = int(np.where(NGUONG == 0.5)[0][0])
print("ngưỡng chung 0.5, trên bộ kiểm tra:")
for ten in ("chon", "tpr", "fpr"):
    a, b = R_kt[0][ten][i50], R_kt[1][ten][i50]
    nhan_ten = {"chon": "tỉ lệ được tuyển", "tpr": "TPR", "fpr": "FPR"}[ten]
    print(f"  {nhan_ten:>16}: A {a:.3f} · B {b:.3f} · chênh {100 * (a - b):+.1f} điểm")

Một ngưỡng chung, áp lên hai phân phối điểm khác nhau, cho hai nhóm những kết quả khác nhau. Đó là ý của
hình 3 trong chương. Với hai giả định trên, nhóm B có tỉ lệ được tuyển thấp hơn, và cả TPR thấp hơn: một
phần người làm tốt của nhóm B bị bỏ lỡ.

## 4. Vì sao các tiêu chí không thể cùng đúng

Bốn tỉ lệ của một ma trận nhầm lẫn không độc lập với nhau. Gọi $p$ là tỉ lệ nền và PPV là giá trị tiên
đoán dương, tức phần làm tốt thật trong số người được chọn. Đếm lại các ô cho ra một đẳng thức, đúng với
mọi ma trận:

$$\text{FPR} = \frac{p}{1-p}\cdot\frac{1-\text{PPV}}{\text{PPV}}\cdot(1-\text{FNR})$$

Kiểm nó trên hai bảng của sách và trên mô hình vừa huấn luyện:

In [ ]:
def fpr_tu_dang_thuc(p, ppv, fnr):
    return p / (1 - p) * (1 - ppv) / ppv * (1 - fnr)


for ten, c in (("bảng 6", cs_a), ("bảng 7", cs_b)):
    ve_phai = fpr_tu_dang_thuc(c["tỉ lệ nền"], c["PPV"], c["FNR"])
    print(f"{ten}: FPR {c['FPR']:.4f} · vế phải {ve_phai:.4f}")
    assert abs(ve_phai - c["FPR"]) < 1e-12
for G, ten in ((0, "mô hình, nhóm A"), (1, "mô hình, nhóm B")):
    r = R_kt[G]
    ve_phai = fpr_tu_dang_thuc(r["nen"], r["ppv"][i50], 1 - r["tpr"][i50])
    print(f"{ten}: FPR {r['fpr'][i50]:.4f} · vế phải {ve_phai:.4f}")
    assert abs(ve_phai - r["fpr"][i50]) < 1e-12

p_a, p_b = R_kt[0]["nen"], R_kt[1]["nen"]
he_so_lech = (p_a / (1 - p_a)) / (p_b / (1 - p_b))
print(f"nếu PPV và FNR bằng nhau ở hai nhóm, FPR của A phải gấp {he_so_lech:.2f} lần FPR của B")

Đẳng thức nói thẳng điều chương nói bằng lời. Một điểm số mà PPV như nhau ở hai nhóm, một dạng của hiệu
chuẩn (calibration) mà COMPAS gần đạt được, và FNR như nhau, thì buộc FPR phải khác nhau đúng theo tỉ số
của $p/(1-p)$, trừ khi hai tỉ lệ nền bằng nhau hoặc dự đoán hoàn hảo. Đó là lý do bản kiểm toán COMPAS
thấy điểm số gần hiệu chuẩn theo chủng tộc mà vẫn vi phạm cân bằng xác suất.

Một xung đột thứ hai còn đơn giản hơn. Tỉ lệ được chọn của một nhóm bằng
$\text{TPR}\cdot p + \text{FPR}\cdot(1-p)$. Nếu cân bằng xác suất đúng, tức TPR và FPR bằng nhau ở hai
nhóm, thì chênh lệch tỉ lệ được chọn bằng $(\text{TPR} - \text{FPR})(p_A - p_B)$. Nó chỉ bằng 0 khi tỉ
lệ nền bằng nhau, hoặc khi TPR bằng FPR, tức mô hình không phân biệt được ai với ai. Vậy khi tỉ lệ nền
khác nhau, ngang bằng nhân khẩu học và cân bằng xác suất không thể cùng đúng ở một mô hình có ích.

### Hai đường ROC

Dời ngưỡng của một nhóm thì điểm vận hành của nhóm ấy trượt dọc đường ROC của nó. Cân bằng xác suất đòi
hai nhóm đứng ở **cùng một điểm** trên mặt phẳng (FPR, TPR).

**Dự đoán:** nhóm nào có đường ROC thấp hơn? Hai đường gặp nhau ở đâu?

In [ ]:
i_eo = int(np.argmin(abs(R_kd[1]["tpr"] - R_kd[0]["tpr"][i50])))  # ngưỡng B cho TPR bằng của A ở 0.5
i_dp = int(np.argmin(abs(R_kd[1]["chon"] - R_kd[0]["chon"][i50])))  # ngưỡng B cho tỉ lệ chọn bằng của A
print(f"giữ ngưỡng A ở 0.5: ngưỡng B cho TPR bằng nhau là {NGUONG[i_eo]:.2f}, "
      f"cho tỉ lệ được chọn bằng nhau là {NGUONG[i_dp]:.2f} (chọn trên bộ kiểm định)")
for ten, i in (("TPR bằng nhau", i_eo), ("tỉ lệ chọn bằng nhau", i_dp)):
    print(f"  {ten}: trên bộ kiểm tra, chênh TPR {100 * (R_kt[0]['tpr'][i50] - R_kt[1]['tpr'][i]):+.1f} điểm, "
          f"chênh tỉ lệ chọn {100 * (R_kt[0]['chon'][i50] - R_kt[1]['chon'][i]):+.1f} điểm, "
          f"chênh FPR {100 * (R_kt[0]['fpr'][i50] - R_kt[1]['fpr'][i]):+.1f} điểm")

fig, ax = plt.subplots(figsize=(5.6, 5.2))
for G, ten in ((0, "nhóm A"), (1, "nhóm B")):
    r = R_kt[G]
    ax.plot(np.r_[1, r["fpr"], 0], np.r_[1, r["tpr"], 0], color=MAU[G], lw=2, label=f"đường ROC, {ten}")
ax.plot(R_kt[0]["fpr"][i50], R_kt[0]["tpr"][i50], "o", color=MAU[0], ms=8)
ax.plot(R_kt[1]["fpr"][i50], R_kt[1]["tpr"][i50], "o", color=MAU[1], ms=8)
ax.plot(R_kt[1]["fpr"][i_eo], R_kt[1]["tpr"][i_eo], "s", color=MAU[1], ms=8, mfc="white", mew=2)
ax.annotate("A ở ngưỡng 0.5", (R_kt[0]["fpr"][i50], R_kt[0]["tpr"][i50]), xytext=(0.05, 0.86),
            fontsize=8, arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax.annotate("B ở ngưỡng 0.5", (R_kt[1]["fpr"][i50], R_kt[1]["tpr"][i50]), xytext=(0.34, 0.30),
            fontsize=8, arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax.annotate(f"B ở ngưỡng {NGUONG[i_eo]:.2f}:\ncùng TPR với A,\nFPR cao hơn", (R_kt[1]["fpr"][i_eo], R_kt[1]["tpr"][i_eo]),
            xytext=(0.55, 0.47), fontsize=8, arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax.plot([0, 1], [0, 1], color=XAM, lw=0.8, ls=":")
ax.text(0.62, 0.56, "đoán ngẫu nhiên", fontsize=8, color=XAM, rotation=38)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("Hai nhóm, hai đường ROC (dữ liệu tổng hợp)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Đường ROC của nhóm B nằm dưới, vì đặc trưng của nhóm B nhiễu hơn. Hạ ngưỡng của B đưa TPR của B lên bằng
A, đổi lại FPR của B tăng: đó đúng là cách điều chỉnh ngưỡng mà chương nêu, và cái giá chương cảnh báo.

Ngưỡng làm bằng TPR và ngưỡng làm bằng tỉ lệ được chọn không có lý do gì phải trùng nhau: tỉ lệ được
chọn trộn TPR với FPR theo tỉ lệ nền, nên hai ngưỡng chỉ gần nhau khi các số hạng ấy tình cờ bù cho
nhau, như ở dữ liệu này. Điều không tình cờ là cả hai ngưỡng đều để FPR của B vượt hẳn FPR của A, nên cân
bằng xác suất vẫn bị vi phạm. Cân bằng xác suất đòi một điểm chung của hai đường, mà ngoài hai đầu mút,
hai đường gần như không gặp nhau: muốn đạt nó bằng ngưỡng, phải kéo cả hai nhóm về gần một góc, nơi mô
hình chọn gần hết hoặc loại gần hết.

## 5. Biên Pareto giữa độ chính xác và chênh lệch

Hình 1 của chương vẽ một biên Pareto giả định: tập các cấu hình mà ở đó một mục tiêu không thể tốt hơn
nếu mục tiêu kia không xấu đi. Sách nói rõ đường cong ấy được dựng để minh hoạ, và biên thật không vẽ
được cho tới khi hiệu năng trên từng nhóm đã được đo. Ở đây thì đo được: mỗi cặp ngưỡng (một cho nhóm A,
một cho nhóm B) là một cấu hình, và lưới có 97 × 97 cặp.

### Dự đoán

Đòi chênh lệch TPR không quá 5 điểm phần trăm thì độ chính xác tổng thể mất khoảng bao nhiêu điểm: dưới
1, vài điểm, hay hơn 10? Còn đòi cân bằng xác suất, tức cả TPR lẫn FPR chênh không quá 2 điểm?

In [ ]:
def luoi(R):
    A, B = R[0], R[1]
    do_cx = (A["dung"][:, None] + B["dung"][None, :]) / (A["n"] + B["n"])
    lech_tpr = abs(A["tpr"][:, None] - B["tpr"][None, :])
    lech_chon = abs(A["chon"][:, None] - B["chon"][None, :])
    lech_xs = np.maximum(lech_tpr, abs(A["fpr"][:, None] - B["fpr"][None, :]))
    return do_cx, lech_tpr, lech_chon, lech_xs


L_kd, L_kt = luoi(R_kd), luoi(R_kt)
NGANG_BANG = sach(5, nguon="5 điểm phần trăm · 2 phần trăm · 0.01") / 100  # định nghĩa 1.2: ε = 5 %


def chon_cap(dieu_kien):
    """Cặp ngưỡng chính xác nhất trên bộ kiểm định thoả điều kiện."""
    a = np.where(dieu_kien, L_kd[0], -1)
    return np.unravel_index(int(a.argmax()), a.shape)


diem_chon = {
    "ngưỡng chung 0.5": (i50, i50),
    "không ràng buộc": chon_cap(np.ones_like(L_kd[0], bool)),
    "TPR chênh ≤ 5 điểm": chon_cap(L_kd[1] <= NGUONG_PHAT_HANH),
    "tỉ lệ chọn chênh ≤ 5 điểm": chon_cap(L_kd[2] <= NGANG_BANG),
    "TPR và FPR chênh ≤ 2 điểm": chon_cap(L_kd[3] <= 0.02),
}
print(f"{'cấu hình':>26} | ngưỡng A, B | trên bộ kiểm tra: độ chính xác · chênh TPR · chênh chọn · chênh FPR")
for ten, (i, j) in diem_chon.items():
    lech_fpr = abs(R_kt[0]["fpr"][i] - R_kt[1]["fpr"][j])
    print(f"{ten:>26} | {NGUONG[i]:.2f}, {NGUONG[j]:.2f} | {100 * L_kt[0][i, j]:.2f} % · "
          f"{100 * L_kt[1][i, j]:.1f} · {100 * L_kt[2][i, j]:.1f} · {100 * lech_fpr:.1f} điểm")
i0, j0 = diem_chon["không ràng buộc"]
for ten in ("TPR chênh ≤ 5 điểm", "TPR và FPR chênh ≤ 2 điểm"):
    i, j = diem_chon[ten]
    do_tren_may(f"độ chính xác mất đi khi đòi {ten}", 100 * (L_kt[0][i0, j0] - L_kt[0][i, j]), "điểm")

In [ ]:
def bien_pareto(do_cx, lech):
    """Các điểm không bị điểm nào vừa chính xác hơn vừa ít chênh hơn trội hơn."""
    thu_tu = np.lexsort((-do_cx.ravel(), lech.ravel()))
    bien, tot_nhat = [], -1.0
    for k in thu_tu:
        if do_cx.ravel()[k] > tot_nhat:
            bien.append(k)
            tot_nhat = do_cx.ravel()[k]
    return np.array(bien)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
for ax, k, ten_truc in ((ax1, 1, "chênh lệch TPR (điểm phần trăm)"),
                        (ax2, 3, "chênh lệch lớn nhất của TPR và FPR (điểm phần trăm)")):
    do_cx, lech = 100 * L_kd[0], 100 * L_kd[k]
    ax.scatter(lech.ravel(), do_cx.ravel(), s=2, color="0.82", label="một cặp ngưỡng (A, B)", rasterized=True)
    b = bien_pareto(do_cx, lech)
    ax.plot(lech.ravel()[b], do_cx.ravel()[b], color=MAU[1 if k == 1 else 2], lw=2, label="biên Pareto")
    ax.set_xlim(50, 0)  # đảo trục như hình 1: chênh lệch thấp nằm bên phải
    ax.set_ylim(55, 77)
    ax.set_xlabel(ten_truc)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, fontsize=8, frameon=False)
for ten, nhan, ax, k, lech_chu in (("không ràng buộc", "A: không ràng buộc", ax1, 1, (8, 10)),
                                   ("TPR chênh ≤ 5 điểm", "B: TPR chênh ≤ 5", ax1, 1, (-20, -34)),
                                   ("không ràng buộc", "A: không ràng buộc", ax2, 3, (8, 10)),
                                   ("TPR và FPR chênh ≤ 2 điểm", "C: TPR và FPR ≤ 2", ax2, 3, (-30, 22))):
    i, j = diem_chon[ten]
    x, yv = 100 * L_kd[k][i, j], 100 * L_kd[0][i, j]
    ax.plot(x, yv, "o", color="black", ms=6)
    ax.annotate(nhan, (x, yv), xytext=lech_chu, textcoords="offset points", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.9),
                arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax1.set_ylabel("độ chính xác tổng thể (%)")
ax1.set_title("Cơ hội bình đẳng")
ax2.set_title("Cân bằng xác suất")
fig.suptitle("Biên Pareto đo được trên bộ kiểm định (dữ liệu tổng hợp)", fontsize=11)
plt.tight_layout()
plt.show()

Hai biên có hình dạng khác nhau, và đó là điểm chính. Với cơ hội bình đẳng, đoạn cuối của biên khá
phẳng: khép khoảng cách TPR chỉ tốn một phần nhỏ độ chính xác, vì hạ ngưỡng của nhóm B đổi được TPR với
một ít dương tính giả. Với cân bằng xác suất, biên rơi mạnh ở đoạn cuối, vì hai đường ROC không gặp nhau
ở vùng có ích.

Sách cảnh báo đúng điều này theo chiều ngược lại: ba điểm của hình 1 không hàm ý rằng huấn luyện không ràng
buộc luôn chính xác nhất, hay rằng chênh lệch bằng không bắt buộc làm giảm độ chính xác. Biên là của
**một** mô hình, **một** bộ dữ liệu và **một** họ can thiệp (ở đây chỉ là dời ngưỡng). Đổi một trong ba,
và biên đổi theo.

Cặp ngưỡng được chọn trên bộ kiểm định và báo trên bộ kiểm tra, nên chênh lệch ở bảng trên có thể vượt
nhẹ giới hạn đã đặt. Thử đặt `HE_SO_DANH_GIA = 1` để thấy nó vượt xa hơn bao nhiêu khi nhóm B nhỏ.

## 6. Cái giá của công bằng: phép tính nhẩm 1.2

Phép tính nhẩm thứ hai đổi đơn vị từ độ chính xác sang tiền. Kịch bản tuyển dụng: nhóm A có TPR 90 phần
trăm, nhóm B 70 phần trăm, độ chính xác tổng thể 85 phần trăm. Các bên liên quan đòi xoá chênh lệch 20
điểm. Theo phản ứng ngưỡng giả định của kịch bản, hạ ngưỡng của B cho tới khi TPR của B đạt 90 phần trăm
làm FPR của B tăng 15 điểm. Lợi ích thay đổi trên mỗi ứng viên nhóm B là

$$\Delta U = \Delta\text{TPR}\cdot p\cdot V_{\text{tốt}} - \Delta\text{FPR}\cdot(1-p)\cdot C_{\text{tồi}}$$

với $p$ là tỉ lệ nền, $V_{\text{tốt}}$ là giá trị một lần tuyển tốt và $C_{\text{tồi}}$ là chi phí một
lần tuyển tồi.

### Dự đoán

Phía được và phía mất, bên nào lớn hơn?

In [ ]:
TPR_A = sach(90, nguon="20 điểm · 90 · 70 · 85") / 100
TPR_B = sach(70, nguon="20 điểm · 90 · 70 · 85") / 100
D_TPR = TPR_A - TPR_B
theo_sach("chênh lệch TPR cần xoá (điểm)", 100 * D_TPR, sach=20, nguon="20 điểm · 90 · 70")
D_FPR = sach(15, nguon="85 · 15 điểm · 20 phần trăm") / 100
NEN = sach(20, nguon="15 điểm · 20 phần trăm · 100,000") / 100
V_TOT = sach(100_000, nguon="20 phần trăm · 100,000 · 50,000")
C_TOI = sach(50_000, nguon="100,000 · 50,000 · 4,000")

duoc = D_TPR * NEN * V_TOT
mat = D_FPR * (1 - NEN) * C_TOI
theo_sach("phía được, mỗi ứng viên nhóm B ($)", duoc, sach=4_000, nguon="50,000 · 4,000 · 6,000")
theo_sach("phía mất, mỗi ứng viên nhóm B ($)", mat, sach=6_000, nguon="4,000 · 6,000 · 2,000")
theo_sach("lỗ ròng, mỗi ứng viên nhóm B ($)", mat - duoc, sach=2_000, nguon="6,000 · 2,000 · 20 phần trăm")

# Sách nói mức lỗ bằng 20 phần trăm lợi ích nền của nhóm B. Lợi ích nền đó phải là:
PHAN_TRAM_LO = sach(20, nguon="2,000 · 20 phần trăm · 30 phần trăm") / 100
loi_ich_nen = (mat - duoc) / PHAN_TRAM_LO
fpr_nen_suy_ra = (TPR_B * NEN * V_TOT - loi_ich_nen) / ((1 - NEN) * C_TOI)
print(f"lợi ích nền của nhóm B: ${loi_ich_nen:,.0f} mỗi ứng viên, ứng với FPR nền {fpr_nen_suy_ra:.0%} (suy ra, sách không in)")

TI_LE_B = sach(30, nguon="20 phần trăm · 30 phần trăm · 600") / 100
theo_sach("lỗ bình quân trên mọi ứng viên ($)", (mat - duoc) * TI_LE_B, sach=600, nguon="30 phần trăm · 600")

Phía mất lớn hơn phía được, nên trong kịch bản này cái giá của công bằng là 20 phần trăm lợi ích của
chính nhóm B. Sách không in FPR nền của nhóm B, nhưng con số 20 phần trăm chỉ khớp khi FPR nền là 10
phần trăm, nên đó là giả định ngầm của kịch bản. Con số bình quân trên mọi ứng viên là $600, còn một tỉ lệ
phần trăm cho toàn bộ thì theo sách cần thêm lợi ích nền của nhóm A, mà kịch bản không cho.

Sách nói thêm: mức lỗ không tự động xảy ra. Khi khoảng cách TPR đến từ một ngưỡng hiệu chuẩn sai, khép
nó có thể làm **tăng** lợi ích ròng. Mức lỗ xuất hiện khi ngưỡng ban đầu đã ở gần điểm tối ưu lợi ích.
Mô hình ở phần 3 cho đo đúng câu này: lấy cùng $V_{\text{tốt}}$ và $C_{\text{tồi}}$, tính lợi ích của nhóm B
tại mọi ngưỡng.

Một điểm mốc để so: nếu điểm số đã hiệu chuẩn, tức điểm 0.4 nghĩa là 40 phần trăm làm tốt, thì chọn một
người có lời khi $s\cdot V_{\text{tốt}} > (1-s)\cdot C_{\text{tồi}}$, tức khi
$s > C_{\text{tồi}}/(V_{\text{tốt}}+C_{\text{tồi}})$.

In [ ]:
nguong_hoa_von = C_TOI / (V_TOT + C_TOI)
print(f"ngưỡng hoà vốn của một điểm số đã hiệu chuẩn: {nguong_hoa_von:.3f}")

r_b = R_kt[1]
loi_ich_b = r_b["tpr"] * r_b["nen"] * V_TOT - r_b["fpr"] * (1 - r_b["nen"]) * C_TOI
i_tot = int(loi_ich_b.argmax())
print(f"nhóm B, lợi ích mỗi ứng viên: ở ngưỡng 0.5 ${loi_ich_b[i50]:,.0f} · ở ngưỡng làm bằng TPR "
      f"({NGUONG[i_eo]:.2f}) ${loi_ich_b[i_eo]:,.0f} · cao nhất ${loi_ich_b[i_tot]:,.0f} ở ngưỡng {NGUONG[i_tot]:.2f}")
thay_doi = loi_ich_b[i_eo] - loi_ich_b[i50]
print(f"khép khoảng cách TPR làm lợi ích của nhóm B {'TĂNG' if thay_doi > 0 else 'GIẢM'} "
      f"${abs(thay_doi):,.0f} mỗi ứng viên ({thay_doi / loi_ich_b[i50]:+.1%}): ngưỡng 0.5 nằm "
      f"{'bên phải' if NGUONG[i_tot] < 0.5 else 'bên trái'} đỉnh")

# Điểm số có hiệu chuẩn trong nhóm B không? So điểm trung bình với tỉ lệ làm tốt thật, theo từng khoảng điểm.
s_b, y_b = diem_kt[g_kt == 1], y_kt[g_kt == 1]
for lo, hi in ((0.2, 0.4), (0.4, 0.6), (0.6, 0.8)):
    m = (s_b >= lo) & (s_b < hi)
    print(f"  nhóm B, điểm trong [{lo}, {hi}): điểm trung bình {s_b[m].mean():.2f}, làm tốt thật {y_b[m].mean():.2f}")

fig, ax = plt.subplots(figsize=(8, 3.9))
ax.plot(NGUONG, loi_ich_b / 1e3, color=MAU[1], lw=2)
for i, nhan, lech_chu in ((i50, "ngưỡng chung 0.5", (10, -30)),
                          (i_eo, "TPR bằng nhóm A", (-80, -40)),
                          (i_tot, "lợi ích cao nhất", (30, 6))):
    ax.plot(NGUONG[i], loi_ich_b[i] / 1e3, "o", color="black", ms=6)
    ax.annotate(nhan, (NGUONG[i], loi_ich_b[i] / 1e3), xytext=lech_chu, textcoords="offset points", fontsize=8,
                arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax.axvline(nguong_hoa_von, color=XAM, lw=1, ls="--")
ax.set_ylim(0, loi_ich_b.max() / 1e3 * 1.15)
ax.text(nguong_hoa_von - 0.01, 0.4, "hoà vốn nếu đã\nhiệu chuẩn", fontsize=8, color=XAM, ha="right")
ax.set_xlabel("ngưỡng của nhóm B")
ax.set_ylabel("lợi ích mỗi ứng viên nhóm B (nghìn $)")
ax.set_title("Lợi ích của nhóm B theo ngưỡng, giá trị của phép tính nhẩm 1.2 (dữ liệu tổng hợp)", fontsize=10)
plt.tight_layout()
plt.show()

Đường lợi ích có một đỉnh. Hạ ngưỡng từ một điểm nằm bên phải đỉnh thì lợi ích tăng, còn hạ quá đỉnh thì
lợi ích giảm. Khép khoảng cách TPR được hay mất, và được mất bao nhiêu, vì vậy tuỳ ngưỡng ban đầu đứng ở
đâu so với đỉnh. Ô trên in ra trường hợp của mô hình này. Phép tính nhẩm 1.2 là trường hợp còn lại: ngưỡng
ban đầu đã gần điểm tối ưu, nên khép khoảng cách thì lỗ. Cả hai là câu "mức lỗ không tự động xảy ra" của
sách, nhìn từ hai phía.

Nếu điểm số hiệu chuẩn trong nhóm B, đỉnh sẽ nằm ở ngưỡng hoà vốn. Bảng ba dòng ở trên so điểm trung bình
với tỉ lệ làm tốt thật trong từng khoảng điểm. Ở khoảng nào hai cột lệch nhau, điểm số không hiệu chuẩn ở
đó, vì mô hình học chung cho hai nhóm có đặc trưng nhiễu khác nhau. Một cái giá tính theo lợi ích vì thế
phụ thuộc cả vào việc điểm số có hiệu chuẩn hay không, thêm một lý do để làm đúng lời sách: đưa cho các bên
liên quan cả đường biên lẫn những giả định đã sinh ra nó.

## Thử thêm

1. Ở phần 2, đổi `thien_lech_lich_su=0.8` thành 0 rồi chạy lại. Hệ số của `ma_hoat_dong` và chênh lệch
   của mô hình "bỏ cột nhóm" còn lại bao nhiêu?
2. Ở phần 3, đặt `lech_nang_luc=0.0` trong `CAU_HINH`, để hai nhóm có cùng tỉ lệ nền như bảng 6 và 7.
   Hệ số in ở cuối phần 4 thành bao nhiêu, và biên của cân bằng xác suất đổi hình ra sao?
3. Ở phần 3, đặt `nhieu_b=0.6`. Khi đặc trưng của B tốt như của A, hai đường ROC còn tách nhau không, và
   cái giá của cân bằng xác suất còn lớn không?

## Tóm lại

* Từ cùng hai ma trận nhầm lẫn, ngang bằng nhân khẩu học thấy 15 điểm, cơ hội bình đẳng thấy 30 điểm, và
  FPR không chênh gì: mỗi định nghĩa là một lựa chọn chính sách.
* Một bộ theo dõi tử tế báo "chưa đủ dữ liệu" cho một tỉ lệ không xác định, không đem nó so với ngưỡng.
* Bỏ thuộc tính nhạy cảm không xoá được tín hiệu của nó: một biến thay thế mang thiên lệch của nhãn lịch
  sử trở lại.
* Khi tỉ lệ nền khác nhau, hiệu chuẩn và cân bằng tỉ lệ lỗi không thể cùng đúng, và ngang bằng nhân khẩu
  học không thể đi cùng cân bằng xác suất ở một mô hình có ích; một đẳng thức đếm ô cho thấy vì sao.
* Cái giá của công bằng là một con số của một mô hình, một bộ dữ liệu và một họ can thiệp: ở phép tính
  nhẩm 1.2 là lỗ $2,000 mỗi ứng viên nhóm B, tức 20 phần trăm lợi ích nền, và nó có thể là lãi nếu ngưỡng
  ban đầu đặt sai.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kỹ thuật có trách nhiệm — viết lại những gì đặc tả bỏ sót](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch15-responsible-engr/), dựng từ chương
[*Responsible Engineering*](https://mlsysbook.ai/vol1/responsible_engr/responsible_engr.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_15_responsible_engr.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.